# 07 Rule-Based Recommendation Engine

Creates simple, transparent recommendations from sleep, stress, mood, workload, and productivity-like features. This rule layer is intentionally easy to replace with an LLM later.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid')
except Exception:
    sns = None

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'ai').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'ai' / 'datasets'
PREPROCESSING_DIR = PROJECT_ROOT / 'ai' / 'preprocessing'
MODEL_DIR = PROJECT_ROOT / 'ai' / 'models'

CSV_TSV_EXTENSIONS = {'.csv', '.tsv'}

def discover_data_files(data_dir=DATA_DIR):
    return sorted(p for p in data_dir.rglob('*') if p.suffix.lower() in CSV_TSV_EXTENSIONS)

def read_dataset(path):
    sep = '\t' if path.suffix.lower() == '.tsv' else ','
    return pd.read_csv(path, sep=sep)

def safe_name(path):
    return path.relative_to(DATA_DIR).as_posix().replace('/', '__')

def display_heading(text):
    try:
        from IPython.display import display, Markdown
        display(Markdown(f'### {text}'))
    except Exception:
        print(f'\n### {text}')

RECOMMENDATION_DIR = PREPROCESSING_DIR / 'recommendations'
RECOMMENDATION_DIR.mkdir(parents=True, exist_ok=True)

def find_column(df, keywords):
    for col in df.columns:
        low = col.lower()
        if any(key in low for key in keywords):
            return col
    return None

def recommend(row, columns):
    recs = []
    sleep = columns.get('sleep')
    stress = columns.get('stress')
    mood = columns.get('mood')
    workload = columns.get('workload')
    productivity = columns.get('productivity')
    if sleep and pd.notna(row.get(sleep)) and row.get(sleep) < 6:
        recs.append('Improve sleep')
    if stress and pd.notna(row.get(stress)) and row.get(stress) >= 7:
        recs.append('Take a break')
    if mood and str(row.get(mood, '')).lower() in {'sadness', 'anger', 'fear', 'disappointment', 'grief', 'negative', 'low'}:
        recs.append('Exercise')
    if productivity and pd.notna(row.get(productivity)) and row.get(productivity) < 50:
        recs.append('Focus session')
    if workload and pd.notna(row.get(workload)) and row.get(workload) > 8:
        recs.append('Reduce workload')
    return recs or ['Maintain current routine']


In [ ]:
for path in discover_data_files():
    display_heading(path.relative_to(DATA_DIR).as_posix())
    try:
        df = read_dataset(path)
        columns = {
            'sleep': find_column(df, ['sleep']),
            'stress': find_column(df, ['stress', 'burnout']),
            'mood': find_column(df, ['mood', 'emotion', 'sentiment']),
            'workload': find_column(df, ['workload', 'work_hours', 'hours worked', 'hours_worked']),
            'productivity': find_column(df, ['productivity', 'performance', 'efficiency']),
        }
        usable = {k: v for k, v in columns.items() if v}
        if not usable:
            print('No recommendation features detected; skipping.')
            continue
        result = df.copy()
        result['recommendations'] = result.apply(lambda row: '; '.join(recommend(row, columns)), axis=1)
        output = RECOMMENDATION_DIR / f'{safe_name(path).rsplit(".", 1)[0]}_recommendations.csv'
        result.to_csv(output, index=False)
        print('Detected feature columns:', usable)
        display(result[list(usable.values()) + ['recommendations']].head())
        print('Saved:', output.relative_to(PROJECT_ROOT))
    except Exception as exc:
        print(f'Recommendation generation failed for {path.name}: {exc}')
